# 00 · Environment, paths and offline smoke test
Run this first. It (1) installs dependencies, (2) creates the folder layout, (3) runs a **synthetic-data smoke test**
of all six unlearning methods (no downloads, ~1 min on CPU) and (4) checks you can fetch the Stable-Diffusion VAE
that FuGuard needs for image proxies.

The smoke test proves the code path runs end-to-end. It says nothing about paper-level numbers.

**Colab:** use `COLAB_START_HERE.ipynb` for the first-time setup (unzip, install, GPU check). This notebook can be re-run any time.

In [ ]:
import os, sys, pathlib, importlib.util, subprocess
_cwd = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents, pathlib.Path("/content/FuGuard_Reproduction")]
             if (p / "fuguard_repro").exists()), None)
if ROOT is None:
    raise RuntimeError("Project not found. On Colab: run COLAB_START_HERE.ipynb first (it unzips to /content/FuGuard_Reproduction).")
os.environ.setdefault("FUGUARD_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))
for _mod in ("geomloss", "diffusers"):          # a Colab runtime restart wipes pip installs -> re-install if missing
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
        break
print("project root:", ROOT)

In [ ]:
import torch
from fuguard_repro.paths import ensure_dirs, ROOT, DATA_DIR, RAW_DIR, TABLES_DIR, FIGURES_DIR
ensure_dirs()
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only - real runs will be very slow)"))
from fuguard_repro import colab
colab.status()

In [ ]:
from fuguard_repro.smoke import run_smoke_test
df = run_smoke_test(verbose=False)
df

### Can this machine download the VAE? (needed once for CIFAR-10/100, SVHN, EuroSAT FuGuard runs)

In [ ]:
from fuguard_repro.config import BASE_IMAGE_CONFIG
from fuguard_repro.proxy import load_vae
vae = load_vae(BASE_IMAGE_CONFIG["vae_id"], "cpu")      # needs internet / HF access the first time
print("VAE loaded:", type(vae).__name__)
del vae